# §12.4.5 — 과제와 위치에 따른 망각 게이트 분포

> 딥러닝 교재 · 3부 12장 4절 5항 (🐍)
> 선행: §12.4.1(셀 경로의 야코비안) · §12.4.4(상수 게이트의 감쇠) · §12.4.6(제거가 아니라 제어)

## 이 노트북이 답하는 질문

1. **게이트는 정말 학습된 기억 제어기로 일하는가?** 과제의 의존 거리에 따라 분포가 갈라지는가.
2. **게이트는 시간적으로도 일하는가?** 기억해야 할 신호 전후로 게이트가 움직이는가.
3. **명제 12.4.1의 감쇠 계산이 실측과 맞는가?**

**예상 실행 시간** CPU 약 90초 (`FAST = True`이면 약 40초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. LSTM — 원리 그대로 구현 (식 12.4.1)

In [ ]:
T_SEQ = 30; M = 24; DIN = 2

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

class LSTM:
    def __init__(self, rn, m=M, d=DIN, forget_bias=1.0):
        k = m + d
        self.W = rn.standard_normal((k, 4*m)) * (1/np.sqrt(k))
        self.bias = np.zeros(4*m)
        self.bias[:m] = forget_bias                     # 망각 게이트 편향 (§12.4.7)
        self.U = rn.standard_normal(m)/np.sqrt(m); self.c0 = np.zeros(1)
        self.params = [self.W, self.bias, self.U, self.c0]
        self.m = m
    def forward(self, X):
        B, T, _ = X.shape; m = self.m
        h = np.zeros((B, m)); c = np.zeros((B, m))
        cache = []
        F = np.zeros((B, T, m))
        for t in range(T):
            z = np.concatenate([h, X[:, t]], axis=1) @ self.W + self.bias
            f = sigmoid(z[:, :m]); i = sigmoid(z[:, m:2*m])
            g = np.tanh(z[:, 2*m:3*m]); o = sigmoid(z[:, 3*m:])
            c_new = f*c + i*g
            h_new = o*np.tanh(c_new)
            cache.append((h.copy(), c.copy(), np.concatenate([h, X[:, t]], axis=1), f, i, g, o, c_new))
            h, c = h_new, c_new
            F[:, t] = f
        self.cache = cache; self.F = F
        out = h @ self.U + self.c0
        self.h_last = h
        return out.ravel()
    def backward(self, dout):
        B = len(dout); m = self.m
        gW = np.zeros_like(self.W); gbias = np.zeros_like(self.bias)
        gU = self.h_last.T @ dout; gc0 = np.array([dout.sum()])
        dh = np.outer(dout, self.U); dc = np.zeros((B, m))
        for t in range(len(self.cache)-1, -1, -1):
            h_prev, c_prev, hx, f, i, g, o, c_new = self.cache[t]
            tc = np.tanh(c_new)
            do = dh * tc
            dc = dc + dh * o * (1 - tc**2)
            df = dc * c_prev; di = dc * g; dg = dc * i
            dz = np.concatenate([df*f*(1-f), di*i*(1-i), dg*(1-g**2), do*o*(1-o)], axis=1)
            gW += hx.T @ dz; gbias += dz.sum(axis=0)
            dhx = dz @ self.W.T
            dh = dhx[:, :m]
            dc = dc * f
        return [gW, gbias, gU, gc0]

def adam(p, g, m, v, t, lr=5e-3):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

---
## 2. 두 과제 — 짧은 의존과 긴 의존

입력은 (값, 표시)의 2차원 열. **짧은 과제**는 마지막 세 값의 합의 부호,
**긴 과제**는 시퀀스 초반 표시 위치 값의 부호를 마지막에 답한다 (운반 거리 $\approx T$).

In [ ]:
def make_task(n, rn, long_dep=True):
    v = rn.standard_normal((n, T_SEQ))
    mk = np.zeros((n, T_SEQ))
    if long_dep:
        p1 = rn.integers(1, 4, n)
        mk[np.arange(n), p1] = 1.0
        y = (v[np.arange(n), p1] > 0).astype(float)
    else:
        y = (v[:, -3:].sum(axis=1) > 0).astype(float)
    X = np.stack([v, mk], axis=2)
    return X, y

def train_task(long_dep, steps=None, seed=0):
    steps = steps or (200 if FAST else 400)
    net = LSTM(np.random.default_rng(seed))
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    rb = np.random.default_rng(300+seed); B = 96
    for t in range(1, steps+1):
        X, y = make_task(B, rb, long_dep)
        logit = net.forward(X)
        p = sigmoid(logit)
        gs = net.backward((p - y)/B)
        for pp, g, m_, v_ in zip(net.params, gs, ms, vs):
            adam(pp, g, m_, v_, t)
    Xev, yev = make_task(1500, np.random.default_rng(SEED+5), long_dep)
    acc = np.mean(((net.forward(Xev)) > 0) == (yev > 0.5))
    return net, acc, (Xev, yev)

net0 = LSTM(np.random.default_rng(0))
Xp, _ = make_task(400, np.random.default_rng(2), True)
_ = net0.forward(Xp); F_init = net0.F.copy()

net_long, acc_long, (Xev_l, yev_l) = train_task(True, seed=0)
net_short, acc_short, _ = train_task(False, seed=0)
print(f"긴 의존 과제 정확도 {acc_long:.3f} | 짧은 의존 과제 정확도 {acc_short:.3f}")

_ = net_long.forward(Xev_l[:400]); F_long = net_long.F.copy()
Xev_s, _ = make_task(400, np.random.default_rng(7), False)
_ = net_short.forward(Xev_s); F_short = net_short.F.copy()

---
## 3. 감쇠의 실측 — 명제 12.4.1

성분별 셀 경로 감도는 $\prod_s f_s$ (명제 12.4.1). 실측 $f$의 궤적으로 이 곱을 계산하고,
상수 게이트 근사 $(\bar f)^k$ (연습 12.4.1)와 비교한다.

In [ ]:
Fl = F_long[:, ::-1, :]                       # 뒤에서부터 거리 k
prod_meas = np.cumprod(Fl, axis=1)            # (B,T,M): 거리 k의 성분별 곱
prod_unit = prod_meas.mean(axis=0)            # (T,M)
fbar_top = F_long.mean(axis=(0, 1))           # 성분별 평균 f
k_top = int(np.argmax(fbar_top))              # 가장 기억형인 성분
k_med = int(np.argsort(fbar_top)[M//2])
ks = np.arange(1, T_SEQ+1)
print(f"기억형 성분의 평균 f = {fbar_top[k_top]:.3f}, 중간 성분 = {fbar_top[k_med]:.3f}")

---
## 4. 교재 그림 — fig_12_4_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 학습 전후 게이트 분포
ax = axes[0]
bins = np.linspace(0, 1, 41)
ax.hist(F_init.ravel(), bins=bins, density=True, alpha=0.55, color=CB[0], label=lab('학습 전', 'init'))
ax.hist(F_long.ravel(), bins=bins, density=True, alpha=0.55, color=CB[5], label=lab('학습 후 (긴 의존)', 'trained (long)'))
ax.set_xlabel(lab('망각 게이트 값 $f$', 'forget gate $f$'))
ax.set_ylabel(lab('밀도', 'density'))
ax.set_title(lab('(a) 게이트 분포 — 두 봉우리로 갈라진다', '(a) gate distribution'), fontsize=10)
ax.legend(fontsize=8)

# (b) 과제별 분포
ax = axes[1]
ax.hist(F_short.ravel(), bins=bins, density=True, alpha=0.55, color=CB[1], label=lab('짧은 의존 과제', 'short-dep task'))
ax.hist(F_long.ravel(), bins=bins, density=True, alpha=0.55, color=CB[5], label=lab('긴 의존 과제', 'long-dep task'))
ax.set_xlabel(lab('망각 게이트 값 $f$', 'forget gate $f$'))
ax.set_ylabel(lab('밀도', 'density'))
ax.set_title(lab('(b) 과제가 요구하는 만큼 기억한다', '(b) task-dependent gating'), fontsize=10)
ax.legend(fontsize=8)

# (c) 표시 신호 전후의 게이트 궤적 (긴 과제, 기억형 성분)
ax = axes[2]
mk_pos = np.argmax(Xev_l[:400, :, 1], axis=1)
tr = np.zeros(T_SEQ); cnt = np.zeros(T_SEQ)
for b in range(400):
    tr += F_long[b, :, k_top]; cnt += 1
ax.plot(range(T_SEQ), tr/cnt, '-', color=CB[5], lw=1.5, label=lab('기억형 성분의 평균 $f$', 'memory unit'))
trm = np.zeros(T_SEQ)
for b in range(400):
    trm += F_long[b, :, k_med]
ax.plot(range(T_SEQ), trm/400, '-', color=CB[1], lw=1.5, label=lab('중간 성분', 'median unit'))
ax.axvspan(1, 3, color=CB[4], alpha=0.15)
ax.text(2, ax.get_ylim()[0]+0.02, lab('표시 구간', 'marker'), color=CB[4], fontsize=8, ha='center')
ax.set_xlabel(lab('시각 $t$', 'time $t$'))
ax.set_ylabel(lab('평균 망각 게이트 $f$', 'mean forget gate'))
ax.set_title(lab('(c) 신호가 들어오면 닫고, 그 뒤로 연다', '(c) gate trajectory'), fontsize=10)
ax.legend(fontsize=8)

# (d) 감쇠: 실측 vs 상수 근사
ax = axes[3]
ax.semilogy(ks, prod_unit[:, k_top], 'o-', color=CB[5], ms=3, label=lab('실측 $\\prod f$ (기억형)', 'measured (memory)'))
ax.semilogy(ks, fbar_top[k_top]**ks, '--', color=CB[5], lw=1, label=lab('$(\\bar f)^k$ 근사', 'constant approx'))
ax.semilogy(ks, prod_unit[:, k_med], 's-', color=CB[1], ms=3, label=lab('실측 (중간)', 'measured (median)'))
ax.semilogy(ks, fbar_top[k_med]**ks, '--', color=CB[1], lw=1)
ax.set_xlabel(lab('시간 거리 $k$', 'distance $k$'))
ax.set_ylabel(lab('셀 경로 감도 $\\prod_s f_s$', 'cell-path sensitivity'))
ax.set_title(lab('(d) 감쇠는 여전히 지수 — 다만 제어된 지수', '(d) controlled decay'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_12_4_5')
plt.show()

> ### 읽는 법
>
> (a) 초기화 시점에 좁게 몰려 있던 게이트가 학습 후 **1 근처의 기억 봉우리**와 낮은 값의
> 즉시-망각 봉우리로 갈라진다. (b) 갈라짐의 정도는 과제가 정한다 — 긴 의존 과제 쪽이 1 근처 질량이 크다.
> (c) 게이트는 시간적으로도 일한다. 표시 신호가 들어오는 구간에서 기억형 성분의 $f$가 치솟아
> 셀을 닫고(보존), 그 정보가 필요 없는 성분은 낮게 머문다.
> (d) 그러나 감쇠는 여전히 지수다. 실측 $\prod f$는 로그 축의 직선에 가깝고 $(\bar f)^k$가 그 뼈대를 준다.
> **게이팅은 소실을 없앤 것이 아니라 감쇠율을 학습 가능한 값으로 바꾼 것이다** (§12.4.6).

---
## 5. 자기 점검

1. (a)의 갈라짐이 forget bias 초기화(기본 1.0)를 0으로 두면 어떻게 달라지는가? 실행해 보라 (§12.4.7).
2. (c)에서 게이트가 표시 **직전**부터 오르는 것처럼 보이면 그 이유는 무엇일 수 있는가?
3. (d)의 실측이 상수 근사보다 아래/위로 벗어나는 구간은 $f$의 시간 상관 때문이다. 옌센 부등식으로 방향을 논하라.
4. 같은 실험을 GRU로 반복하면 (a)에서 무엇이 $f$의 역할을 하는가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `forget_bias` | 1절 | 1.0 | 0/−2로 낮춰 초기화 함정 관찰 |
| `T_SEQ` | 1절 | 30 | 운반 거리 |
| `steps` | 2절 | 400 | 학습 길이 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")